# แบบฝึกปฏิบัติการ: การวิเคราะห์ข้อมูลและการสร้างแบบจำลองเบื้องต้น

### กรณีศึกษา: ข้อมูลเบาะแส/เรื่องร้องเรียนยาเสพติด สำนักงานคณะกรรมการป้องกันและปราบปรามยาเสพติด (ป.ป.ส.)

เอกสารประกอบการฝึกอบรมฉบับนี้จัดทำขึ้นเพื่อให้ผู้เข้ารับการอบรมได้ฝึกปฏิบัติการวิเคราะห์ข้อมูลภาครัฐ
ด้วยภาษา Python บนระบบ Google Colab ครอบคลุมทั้งการนำเสนอข้อมูลด้วยแผนภาพ (Visualization)
และการสร้างแบบจำลอง (Clustering และ Regression)

เอกสารนี้ออกแบบสำหรับผู้เริ่มต้น โดยมีตัวอย่างพร้อมคำอธิบายประกอบในทุกขั้นตอน
และมีแนวทางประกอบการทำแบบฝึกปฏิบัติในแต่ละข้อ

---

### แผนการฝึกอบรม (เรียงจากระดับพื้นฐานไปสู่ระดับสูง)

| ส่วน | หัวข้อ | ระดับ |
|---|---|---|
| 0 | ทำความรู้จักข้อมูลและคำสั่งพื้นฐานของ pandas | บรรยาย |
| 1 | บทเรียนการนำเสนอข้อมูล — Histogram / Bar / Line / แผนที่ | บรรยาย |
| 2 | แบบฝึกปฏิบัติการนำเสนอข้อมูล | พื้นฐาน–สูง |
| 3 | บทเรียนการสร้างแบบจำลอง — Clustering และ Regression | บรรยาย |
| 4 | แบบฝึกปฏิบัติการสร้างแบบจำลอง | ระดับสูง |

**วิธีการทำแบบฝึกปฏิบัติ:** ในแต่ละข้อจะมีตัวอย่างเริ่มต้นในเซลล์โค้ดให้แล้ว ขอให้ผู้เข้ารับการอบรม
ดำเนินการเขียนโค้ดต่อด้วยตนเอง หากพบปัญหา สามารถเปิดดู **แนวโค้ด** (โครงสำหรับเติมช่องว่าง) ก่อน
แล้วจึงเปิดดู **เฉลย** เมื่อจำเป็น

## ขั้นเตรียมความพร้อม (โปรดดำเนินการรัน 3 เซลล์ต่อไปนี้ก่อนทุกครั้ง)

In [ ]:
# 1) ติดตั้งไลบรารีที่จำเป็น (ระบบ Colab มีให้บางส่วนแล้ว ดำเนินการเพื่อความครบถ้วน)
!pip install -q pandas numpy plotly scikit-learn

In [ ]:
# 2) ดาวน์โหลดชุดข้อมูล 3 ไฟล์จาก GitHub มายังระบบ Colab (/content/)
#    - oncb_verify_monthly.csv   : ข้อมูลเบาะแสรายเดือน (ไฟล์หลัก)
#    - th_provinces.geojson      : ขอบเขตจังหวัด (สำหรับการจัดทำแผนที่)
#    - th_province_population.csv : จำนวนประชากรรายจังหวัด
!wget -q -O oncb_verify_monthly.csv    https://raw.githubusercontent.com/E27-25/thai-gov-complaint-viz/main/%E0%B8%9B%E0%B8%9B%E0%B8%AA/data/oncb_verify_monthly.csv
!wget -q -O th_provinces.geojson       https://raw.githubusercontent.com/E27-25/thai-gov-complaint-viz/main/%E0%B8%9B%E0%B8%9B%E0%B8%AA/data/th_provinces.geojson
!wget -q -O th_province_population.csv  https://raw.githubusercontent.com/E27-25/thai-gov-complaint-viz/main/%E0%B8%9B%E0%B8%9B%E0%B8%AA/data/th_province_population.csv
print("ดาวน์โหลดชุดข้อมูลเรียบร้อยแล้ว")

In [ ]:
# 3) เรียกใช้ไลบรารีทั้งหมดที่ใช้ในเอกสารนี้ (ดำเนินการเพียงครั้งเดียว)
import json
import pandas as pd
import numpy as np
import plotly.express as px          # จัดทำกราฟอย่างกระชับ (ใช้งานเป็นหลัก)
import plotly.graph_objects as go    # จัดทำกราฟที่ต้องปรับแต่งเพิ่มเติม

from sklearn.preprocessing import StandardScaler   # ปรับมาตราส่วนของตัวแปร
from sklearn.cluster       import KMeans           # แบบจำลองการจัดกลุ่ม
from sklearn.decomposition import PCA              # ลดมิติข้อมูลเพื่อการนำเสนอ
from sklearn.linear_model  import LinearRegression # แบบจำลองการถดถอย
from sklearn.metrics       import r2_score, mean_absolute_error  # การประเมินผลแบบจำลอง

print("เรียกใช้ไลบรารีครบถ้วนแล้ว")

## ส่วนที่ 0 — ทำความรู้จักข้อมูล (โปรดศึกษาก่อนเริ่มปฏิบัติ)

**ลักษณะของข้อมูล**  
สำนักงาน ป.ป.ส. (ONCB) ได้เผยแพร่สถิติเบาะแสและเรื่องร้องเรียนเกี่ยวกับยาเสพติดที่ประชาชนแจ้งเข้ามา
โดยเจ้าหน้าที่ได้ดำเนินการตรวจสอบข้อเท็จจริง และรายงานผลเป็นรายเดือน จำแนกตามจังหวัด

**ระดับความละเอียดของข้อมูล**  
> ข้อมูล 1 แถว หมายถึงข้อมูลของ 1 จังหวัด ในช่วง 1 เดือน (เช่น กรุงเทพมหานคร เดือนมกราคม 2569)

**คอลัมน์ที่ใช้ในการวิเคราะห์**

| คอลัมน์ | ความหมาย | ตัวอย่าง |
|---|---|---|
| `PROV_NAME` | ชื่อจังหวัด | กรุงเทพมหานคร |
| `NEWS_YEAR` | ปี (พุทธศักราช) | 2569 |
| `NEWS_MONTH_CODE` | เดือน (1–12) | 6 |
| `COMPLAIN_ALL` | จำนวนเบาะแสทั้งหมด (เรื่อง) | 245 |
| `ALL1` | ตรวจสอบแล้วพบพฤติการณ์ | 80 |
| `ALL2` | ตรวจสอบแล้วไม่พบพฤติการณ์ | 120 |
| `ALL3` | ไม่สามารถพิสูจน์ได้ | 30 |
| `ALL123` | จำนวนที่ตรวจสอบแล้วทั้งหมด (= ALL1 + ALL2 + ALL3) | 230 |

หมายเหตุ: ค่าที่ใช้บ่อยคือ **found_rate** = ALL1 / ALL123 หมายถึงสัดส่วนที่ตรวจสอบแล้วพบพฤติการณ์

### คำสั่งพื้นฐานของ pandas ที่ควรทราบ

ก่อนการวิเคราะห์ทุกครั้ง ควรทำความรู้จักข้อมูลด้วยคำสั่งพื้นฐาน 5 คำสั่งต่อไปนี้

| คำสั่ง | หน้าที่ |
|---|---|
| `df.head()` | แสดงข้อมูล 5 แถวแรก เพื่อดูลักษณะข้อมูล |
| `df.shape` | แสดงขนาดตาราง (จำนวนแถว, จำนวนคอลัมน์) |
| `df.info()` | แสดงชนิดข้อมูลของแต่ละคอลัมน์ และจำนวนข้อมูลที่ไม่ว่าง |
| `df.isna().sum()` | นับจำนวนค่าว่าง (NaN) ในแต่ละคอลัมน์ |
| `df.dropna()` | ตัดแถวที่มีค่าว่างออกจากตาราง |

ลำดับถัดไปเป็นการใช้งานทีละคำสั่งกับข้อมูลจริง

**คำสั่งที่ 1–2: `read_csv` และ `head()`** — อ่านไฟล์ข้อมูล แล้วแสดง 5 แถวแรก

In [ ]:
# อ่านไฟล์ CSV (กำหนด encoding='utf-8-sig' เพื่อให้แสดงภาษาไทยได้ถูกต้อง)
df = pd.read_csv('oncb_verify_monthly.csv', encoding='utf-8-sig')
df.columns = df.columns.str.strip()      # ตัดช่องว่างหน้า-หลังของชื่อคอลัมน์

df.head()      # แสดง 5 แถวแรก (สามารถระบุจำนวนได้ เช่น df.head(10))

**คำสั่งที่ 3: `shape` และ `info()`** — แสดงขนาดตาราง และชนิดข้อมูลของแต่ละคอลัมน์

In [ ]:
print('ขนาดตาราง (แถว, คอลัมน์):', df.shape)
print()
df.info()      # แสดงชนิดข้อมูล (int/float/object) และจำนวนข้อมูลที่ไม่ว่างของแต่ละคอลัมน์

**คำสั่งที่ 4: `isna().sum()`** — ตรวจสอบว่าแต่ละคอลัมน์มีค่าว่าง (NaN) จำนวนเท่าใด  
> หากมีค่าว่างจำนวนมาก จะส่งผลต่อความถูกต้องของการคำนวณ จึงต้องจัดการก่อน

In [ ]:
# นับจำนวนค่าว่างในแต่ละคอลัมน์ (แสดงเฉพาะคอลัมน์ที่มีค่าว่าง)
missing = df.isna().sum()
print('จำนวนค่าว่างรวมทั้งตาราง:', int(missing.sum()))
print(missing[missing > 0] if (missing > 0).any() else '→ ไม่พบค่าว่าง ข้อมูลมีความสมบูรณ์')

**คำสั่งที่ 5: `dropna()` และการเตรียมข้อมูล** — ตัดแถวที่ข้อมูลสำคัญว่าง แล้วปรับชนิดข้อมูลให้พร้อมใช้งาน

> การระบุ `subset=[...]` หมายถึงการตัดเฉพาะแถวที่คอลัมน์สำคัญเหล่านั้นมีค่าว่าง มิใช่ตัดทุกกรณีที่พบค่าว่าง

In [ ]:
# ตัดแถวที่คอลัมน์สำคัญมีค่าว่าง (กรณีนี้ไม่มีแถวที่ต้องตัด แต่กำหนดไว้เพื่อรองรับข้อมูลชุดใหม่ที่อาจมีค่าว่าง)
df = df.dropna(subset=['PROV_NAME', 'NEWS_YEAR', 'NEWS_MONTH_CODE'])

# ปรับชนิดข้อมูลให้ถูกต้อง: ปีและเดือนเป็นจำนวนเต็ม, ชื่อจังหวัดตัดช่องว่าง
df['NEWS_YEAR']       = df['NEWS_YEAR'].astype(int)
df['NEWS_MONTH_CODE'] = df['NEWS_MONTH_CODE'].astype(int)
df['PROV_NAME']       = df['PROV_NAME'].str.strip()

# สร้างคอลัมน์วันที่ 'date' (แปลงปีพุทธศักราชเป็นคริสต์ศักราชโดยลบ 543) สำหรับการจัดทำกราฟแนวโน้ม
df['date'] = pd.to_datetime(dict(year=df['NEWS_YEAR'] - 543,
                                 month=df['NEWS_MONTH_CODE'], day=1))

print('ขนาดตารางภายหลังการเตรียมข้อมูล:', df.shape)
df.head(3)

**การสรุปภาพรวมข้อมูล** — ใช้คำสั่ง `describe()`, `nunique()` และ `unique()` เพื่อดูสถิติเบื้องต้น

In [ ]:
# describe() แสดงสถิติสรุปของคอลัมน์ตัวเลข (ค่าเฉลี่ย ค่าต่ำสุด ค่าสูงสุด เป็นต้น)
print('สถิติสรุปของ COMPLAIN_ALL (ต่อจังหวัดต่อเดือน):')
print(df['COMPLAIN_ALL'].describe().round(1))
print()
# nunique() นับจำนวนค่าที่ไม่ซ้ำ, unique() แสดงรายการค่าที่ไม่ซ้ำ
print('จำนวนจังหวัด :', df['PROV_NAME'].nunique())
print('ปี (พ.ศ.)    :', sorted(df['NEWS_YEAR'].unique()))
print('จำนวนเบาะแสรวมทั้งหมด :', f"{int(df['COMPLAIN_ALL'].sum()):,} เรื่อง")

> **ข้อสังเกตและการตีความ**  
> ชุดข้อมูลครอบคลุมครบ 77 จังหวัด และมีข้อมูลต่อเนื่อง 10 ปี (พ.ศ. 2560–2569) ซึ่งเพียงพอต่อการวิเคราะห์
> ค่ากลาง (median) ของจำนวนเบาะแสต่อจังหวัดต่อเดือนอยู่ที่ประมาณ 16 เรื่อง แต่ค่าสูงสุดถึงประมาณ 495 เรื่อง
> แสดงว่าข้อมูลมีการกระจายแบบเบ้ขวา (ส่วนใหญ่มีค่าน้อย มีบางจังหวัด/เดือนที่มีค่าสูงมาก) ซึ่งเป็นลักษณะปกติของข้อมูลเชิงนับจำนวน

## ส่วนที่ 1 — บทเรียนการนำเสนอข้อมูล (ศึกษาตัวอย่างก่อนทำแบบฝึกปฏิบัติ)

เอกสารนี้ใช้ไลบรารี **Plotly Express (`px`)** เป็นหลัก เนื่องจากเขียนได้กระชับและได้กราฟที่โต้ตอบได้
หลักการคือ **จัดข้อมูลให้อยู่ในรูปตารางที่พร้อมนำเสนอ แล้วเรียกใช้คำสั่ง `px.<ชนิดกราฟ>(...)`**

### ตัวอย่างที่ 1 — Histogram (การกระจายของข้อมูล)

Histogram ใช้พิจารณาว่าค่าส่วนใหญ่ของข้อมูลกระจุกตัวอยู่ในช่วงใด โดยใช้คำสั่ง `px.histogram(ตาราง, x='คอลัมน์')`

In [ ]:
# การกระจายของจำนวนเบาะแสต่อจังหวัดต่อเดือน
fig = px.histogram(df, x='COMPLAIN_ALL', nbins=50,
                   title='การกระจายของจำนวนเบาะแส (ต่อจังหวัดต่อเดือน)')
fig.update_layout(xaxis_title='จำนวนเบาะแส (เรื่อง)', yaxis_title='ความถี่ (จำนวนแถว)')
fig.show()

> **ข้อสังเกตและการตีความ**  
> แท่งความถี่กระจุกตัวทางด้านซ้าย (ค่าน้อย) แล้วลดหลั่นไปทางขวา แสดงว่าข้อมูลมีการกระจายแบบเบ้ขวา
> หมายความว่าจังหวัด/เดือนส่วนใหญ่มีเบาะแสไม่มาก แต่มีบางกรณีที่มีจำนวนสูงเป็นพิเศษ (มักเป็นจังหวัดขนาดใหญ่)
> ผลลัพธ์นี้สอดคล้องกับธรรมชาติของข้อมูลเชิงนับจำนวนซึ่งมักมีการกระจายแบบเบ้ขวา

### ตัวอย่างที่ 2 — Bar chart (การเปรียบเทียบระหว่างกลุ่ม)

Bar chart ใช้เปรียบเทียบค่าระหว่างกลุ่ม โดยต้องสรุปข้อมูลให้อยู่ในรูปตารางย่อยก่อนจึงนำเสนอ

In [ ]:
# สรุปผลการตรวจสอบทั้ง 3 ประเภทให้อยู่ในรูปตารางย่อย
result = pd.DataFrame({
    'ผลการตรวจสอบ': ['พบพฤติการณ์', 'ไม่พบพฤติการณ์', 'พิสูจน์ไม่ได้'],
    'จำนวน':        [df['ALL1'].sum(), df['ALL2'].sum(), df['ALL3'].sum()],
})

fig = px.bar(result, x='ผลการตรวจสอบ', y='จำนวน', color='ผลการตรวจสอบ', text='จำนวน',
             title='ผลการตรวจสอบเบาะแส (รวมทั้งประเทศ)')
fig.show()

> **ข้อสังเกตและการตีความ**  
> ประเภท 'ไม่พบพฤติการณ์' มีจำนวนสูงสุด (ประมาณร้อยละ 42) รองลงมาคือ 'พบพฤติการณ์' (ประมาณร้อยละ 33) และ 'พิสูจน์ไม่ได้' (ประมาณร้อยละ 25)
> แสดงว่าเบาะแสที่ได้รับแจ้งส่วนใหญ่เมื่อตรวจสอบแล้วไม่พบพฤติการณ์ ซึ่งสอดคล้องกับลักษณะงานเบาะแสที่ต้องผ่านการคัดกรอง
> โดยพบพฤติการณ์จริงประมาณ 1 ใน 3 ของที่ตรวจสอบทั้งหมด

### ตัวอย่างที่ 3 — Line chart (แนวโน้มตามช่วงเวลา)

Line chart ใช้พิจารณาแนวโน้มเมื่อแกนนอนเป็นช่วงเวลา โดยต้องสรุปข้อมูลเป็นรายเดือนก่อนจึงนำเสนอ

In [ ]:
# สรุปจำนวนเบาะแสทั้งประเทศเป็นรายเดือน (เรียงตามวันที่)
monthly = df.groupby('date', as_index=False)['COMPLAIN_ALL'].sum()

fig = px.line(monthly, x='date', y='COMPLAIN_ALL',
              title='แนวโน้มจำนวนเบาะแสรายเดือน (ทั้งประเทศ)')
fig.update_layout(xaxis_title='เดือน', yaxis_title='จำนวนเบาะแส (เรื่อง)')
fig.show()

> **ข้อสังเกตและการตีความ**  
> เส้นกราฟมีแนวโน้มเพิ่มขึ้นอย่างต่อเนื่อง จากค่าเฉลี่ยประมาณ 1,288 เรื่องต่อเดือนในปีแรก เป็นประมาณ 3,278 เรื่องต่อเดือนในปีล่าสุด หรือเพิ่มขึ้นประมาณ 2.5 เท่า
> และเพิ่มขึ้นอย่างชัดเจนในช่วงปีท้าย ๆ
> ปรากฏการณ์นี้อาจสะท้อนถึงช่องทางการแจ้งเบาะแสที่สะดวกขึ้น หรือมาตรการป้องกันและปราบปรามที่เข้มข้นขึ้น ทั้งนี้ควรพิจารณาข้อมูลประกอบอื่นเพิ่มเติม

### ตัวอย่างที่ 4 — แผนที่ Choropleth (ระดับสูง)

แผนที่ที่ระบายสีตามจังหวัด เรียกว่า Choropleth ซึ่งต้องใช้ไฟล์ GeoJSON ที่จัดเก็บรูปร่างขอบเขตของแต่ละจังหวัด

**ประเด็นสำคัญคือการจับคู่ชื่อจังหวัดจากสองแหล่งให้ตรงกัน**
- ในไฟล์ GeoJSON ชื่อจังหวัดภาษาไทยอยู่ที่ `properties.pro_th` จึงกำหนด `featureidkey='properties.pro_th'`
- ในตารางข้อมูล ชื่อจังหวัดอยู่ที่คอลัมน์ `PROV_NAME` จึงกำหนด `locations='PROV_NAME'`

ขั้นตอนประกอบด้วย 3 ขั้น: (1) อ่านไฟล์ GeoJSON (2) สรุปข้อมูลรายจังหวัด (3) จัดทำกราฟด้วย `px.choropleth`

In [ ]:
# (1) อ่านไฟล์ GeoJSON (อยู่ในรูป dictionary) แล้วตรวจสอบ property ที่มี
with open('th_provinces.geojson', encoding='utf-8') as f:
    geojson = json.load(f)

print('property ในแต่ละจังหวัด :', list(geojson['features'][0]['properties'].keys()))
print('ชื่อจังหวัดแรกในไฟล์    :', geojson['features'][0]['properties']['pro_th'])

In [ ]:
# (2) สรุปจำนวนเบาะแสรวมรายจังหวัด (1 แถว ต่อ 1 จังหวัด)
prov = df.groupby('PROV_NAME', as_index=False)['COMPLAIN_ALL'].sum()

# (3) จัดทำแผนที่ระบายสี
fig = px.choropleth(
    prov,
    geojson=geojson,                     # แม่แบบแผนที่
    featureidkey='properties.pro_th',    # ชื่อจังหวัดในไฟล์ GeoJSON
    locations='PROV_NAME',               # ชื่อจังหวัดในตารางข้อมูล
    color='COMPLAIN_ALL',                # ค่าที่ใช้ระบายสี
    color_continuous_scale='OrRd',       # ชุดสี (อ่อนไปเข้ม)
)
fig.update_geos(fitbounds='locations', visible=False)   # แสดงเฉพาะประเทศไทย
fig.update_layout(title='แผนที่จำนวนเบาะแสยาเสพติดรายจังหวัด',
                  height=600, margin=dict(t=50, l=0, r=0, b=0))
fig.show()

> **ข้อสังเกตและการตีความ**  
> กรุงเทพมหานครมีจำนวนสูงสุด (ประมาณ 26,000 เรื่อง) รองลงมาเป็นจังหวัดปริมณฑลและจังหวัดขนาดใหญ่ (ปทุมธานี ชลบุรี) แสดงว่าเบาะแสกระจุกตัวในเขตเมืองใหญ่
> ผลลัพธ์นี้สอดคล้องกับเหตุผลด้านจำนวนประชากรและกิจกรรมทางเศรษฐกิจที่หนาแน่น
> อนึ่ง ข้อมูลนี้ยังมิได้ปรับด้วยจำนวนประชากร ซึ่งจังหวัดขนาดใหญ่ย่อมมีจำนวนสูงเป็นปกติ ประเด็นนี้จะได้รับการปรับในแบบฝึกปฏิบัติที่ 5 (per-capita)

---
# ส่วนที่ 2 — แบบฝึกปฏิบัติการนำเสนอข้อมูล

แบบฝึกปฏิบัติเรียงจากระดับพื้นฐานไปสู่ระดับสูง โดยในเซลล์โค้ดมีตัวอย่างเริ่มต้นให้แล้ว ขอให้ดำเนินการต่อด้วยตนเอง

## แบบฝึกปฏิบัติที่ 1 — Histogram (ระดับพื้นฐาน)

**โจทย์:** จัดทำ Histogram แสดงการกระจายของ `ALL1` (จำนวนที่ตรวจสอบแล้วพบพฤติการณ์) ต่อจังหวัดต่อเดือน

**แนวทาง:** ใช้รูปแบบเดียวกับตัวอย่างที่ 1 โดยเปลี่ยนคอลัมน์ `x` จาก `COMPLAIN_ALL` เป็น `ALL1`

In [ ]:
# เริ่มต้นจากตัวอย่างนี้ แล้วปรับให้ตรงตามโจทย์
fig = px.histogram(df, x='COMPLAIN_ALL', nbins=50)   # ปรับ 'COMPLAIN_ALL' เป็น 'ALL1'
fig.show()

<details>
<summary><b>แนวโค้ด (โครงสำหรับเติมช่องว่าง ____ ให้สมบูรณ์)</b></summary>

```python
fig = px.histogram(df, x='____', nbins=50,
                   title='การกระจายของจำนวนที่พบพฤติการณ์ (ALL1)')
fig.update_layout(xaxis_title='จำนวนที่พบพฤติการณ์', yaxis_title='ความถี่')
fig.show()
```

</details>

<details>
<summary><b>เฉลย (โค้ดฉบับสมบูรณ์)</b></summary>

```python
fig = px.histogram(df, x='ALL1', nbins=50,
                   title='การกระจายของจำนวนที่พบพฤติการณ์ (ALL1)')
fig.update_layout(xaxis_title='จำนวนที่พบพฤติการณ์', yaxis_title='ความถี่')
fig.show()
```

</details>

## แบบฝึกปฏิบัติที่ 2 — Bar chart: 10 จังหวัดสูงสุด (ระดับพื้นฐาน)

**โจทย์:** ค้นหา 10 จังหวัดที่มีจำนวนเบาะแสรวมสูงสุด แล้วจัดทำกราฟแท่งแนวนอน

**แนวทาง:** ใช้ `groupby` สรุปรายจังหวัด → `sort_values` เรียงจากมากไปน้อย → `head(10)` → `px.bar(orientation='h')`

In [ ]:
# เริ่มต้นจากการสรุปข้อมูลรายจังหวัด (บรรทัดนี้จัดเตรียมให้แล้ว)
top10 = df.groupby('PROV_NAME')['COMPLAIN_ALL'].sum().reset_index()
print(top10.head())

# ดำเนินการต่อ: เรียงจากมากไปน้อย เลือก 10 อันดับแรก แล้วจัดทำ px.bar(top10, x=..., y=..., orientation='h')

<details>
<summary><b>แนวโค้ด (โครงสำหรับเติมช่องว่าง ____ ให้สมบูรณ์)</b></summary>

```python
top10 = (df.groupby('PROV_NAME')['COMPLAIN_ALL'].sum()
           .sort_values(ascending=____)
           .head(____)
           .reset_index())

fig = px.bar(top10, x='COMPLAIN_ALL', y='____', orientation='h',
             color='COMPLAIN_ALL', color_continuous_scale='OrRd',
             title='10 จังหวัดที่มีเบาะแสสูงสุด')
fig.update_layout(yaxis=dict(categoryorder='total ascending'))
fig.show()
```

</details>

<details>
<summary><b>เฉลย (โค้ดฉบับสมบูรณ์)</b></summary>

```python
top10 = (df.groupby('PROV_NAME')['COMPLAIN_ALL'].sum()
           .sort_values(ascending=False)
           .head(10)
           .reset_index())

fig = px.bar(top10, x='COMPLAIN_ALL', y='PROV_NAME', orientation='h',
             color='COMPLAIN_ALL', color_continuous_scale='OrRd',
             title='10 จังหวัดที่มีเบาะแสสูงสุด')
fig.update_layout(yaxis=dict(categoryorder='total ascending'))
fig.show()
```

</details>

## แบบฝึกปฏิบัติที่ 3 — Line chart: แนวโน้มของกรุงเทพมหานคร (ระดับกลาง)

**โจทย์:** จัดทำกราฟเส้นแสดงแนวโน้มจำนวนเบาะแสรายเดือนเฉพาะจังหวัดกรุงเทพมหานคร

**แนวทาง:** คัดกรองเฉพาะกรุงเทพมหานคร → เรียงตามคอลัมน์ `date` → `px.line(x='date', y='COMPLAIN_ALL')`

> การคัดกรองแถว: `df[df['PROV_NAME'] == 'กรุงเทพมหานคร']`

In [ ]:
# คัดกรองเฉพาะกรุงเทพมหานคร (จัดเตรียมให้แล้ว)
bkk = df[df['PROV_NAME'] == 'กรุงเทพมหานคร']

# ดำเนินการต่อ: เรียงตามวันที่ด้วย .sort_values('date') แล้วจัดทำ px.line(bkk, x='date', y='COMPLAIN_ALL')

<details>
<summary><b>แนวโค้ด (โครงสำหรับเติมช่องว่าง ____ ให้สมบูรณ์)</b></summary>

```python
bkk = df[df['PROV_NAME'] == '____'].sort_values('date')

fig = px.line(bkk, x='____', y='____',
              title='แนวโน้มเบาะแสรายเดือน — กรุงเทพมหานคร')
fig.update_layout(xaxis_title='เดือน', yaxis_title='จำนวนเบาะแส')
fig.show()
```

</details>

<details>
<summary><b>เฉลย (โค้ดฉบับสมบูรณ์)</b></summary>

```python
bkk = df[df['PROV_NAME'] == 'กรุงเทพมหานคร'].sort_values('date')

fig = px.line(bkk, x='date', y='COMPLAIN_ALL',
              title='แนวโน้มเบาะแสรายเดือน — กรุงเทพมหานคร')
fig.update_layout(xaxis_title='เดือน', yaxis_title='จำนวนเบาะแส')
fig.show()
```

</details>

## แบบฝึกปฏิบัติที่ 4 — แผนที่: อัตราการพบพฤติการณ์รายจังหวัด (ระดับสูง)

**โจทย์:** จัดทำแผนที่ระบายสีตามค่า **found_rate = ALL1 / ALL123** (สัดส่วนที่ตรวจสอบแล้วพบพฤติการณ์) ของแต่ละจังหวัด

**แนวทาง:** สรุป `ALL1` และ `ALL123` รายจังหวัด → คำนวณ `found_rate` → `px.choropleth(color='found_rate')`

> ใช้ตัวแปร `geojson` ที่อ่านไว้แล้วในตัวอย่างที่ 4 ได้ทันที (โปรดรันตัวอย่างที่ 4 ก่อน)

In [ ]:
# สรุปข้อมูลรายจังหวัด (จัดเตรียมให้แล้ว — ต้องรันตัวอย่างที่ 4 เพื่อให้มีตัวแปร geojson ก่อน)
rate = df.groupby('PROV_NAME', as_index=False)[['ALL1', 'ALL123']].sum()

# ดำเนินการต่อ: rate['found_rate'] = rate['ALL1'] / rate['ALL123']
# แล้วจัดทำ px.choropleth(rate, geojson=geojson, featureidkey='properties.pro_th',
#                          locations='PROV_NAME', color='found_rate')

<details>
<summary><b>แนวโค้ด (โครงสำหรับเติมช่องว่าง ____ ให้สมบูรณ์)</b></summary>

```python
rate = df.groupby('PROV_NAME', as_index=False)[['ALL1', 'ALL123']].sum()
rate['found_rate'] = rate['____'] / rate['____']

fig = px.choropleth(rate, geojson=geojson,
                    featureidkey='properties.pro_th', locations='____',
                    color='____', color_continuous_scale='Greens')
fig.update_geos(fitbounds='locations', visible=False)
fig.update_layout(title='อัตราการตรวจพบพฤติการณ์รายจังหวัด', height=600,
                  margin=dict(t=50, l=0, r=0, b=0))
fig.show()
```

</details>

<details>
<summary><b>เฉลย (โค้ดฉบับสมบูรณ์)</b></summary>

```python
rate = df.groupby('PROV_NAME', as_index=False)[['ALL1', 'ALL123']].sum()
rate['found_rate'] = rate['ALL1'] / rate['ALL123']

fig = px.choropleth(rate, geojson=geojson,
                    featureidkey='properties.pro_th', locations='PROV_NAME',
                    color='found_rate', color_continuous_scale='Greens')
fig.update_geos(fitbounds='locations', visible=False)
fig.update_layout(title='อัตราการตรวจพบพฤติการณ์รายจังหวัด', height=600,
                  margin=dict(t=50, l=0, r=0, b=0))
fig.show()
```

</details>

## แบบฝึกปฏิบัติที่ 5 — แผนที่อัตราต่อประชากร (per-capita) (ระดับสูง)

เนื่องจากจังหวัดขนาดใหญ่มีประชากรมาก จำนวนเบาะแสย่อมสูงตามไปด้วย การเปรียบเทียบอย่างเป็นธรรมจึงควรปรับด้วยจำนวนประชากร

**โจทย์:** จัดทำแผนที่แสดงจำนวนเบาะแสต่อประชากร 100,000 คน

**แนวทาง:** อ่านข้อมูลประชากร → รวม (`merge`) กับข้อมูลเบาะแสรายจังหวัดด้วยคีย์ `PROV_NAME` → คำนวณ `rate = COMPLAIN_ALL / population * 100000` → จัดทำแผนที่โดยกำหนด `color='rate'`

In [ ]:
# อ่านข้อมูลประชากรและสรุปเบาะแสรายจังหวัด (จัดเตรียมให้แล้ว)
pop  = pd.read_csv('th_province_population.csv', encoding='utf-8-sig')
prov = df.groupby('PROV_NAME', as_index=False)['COMPLAIN_ALL'].sum()

# ดำเนินการต่อ: prov = prov.merge(pop, on='PROV_NAME')
# prov['rate'] = prov['COMPLAIN_ALL'] / prov['population'] * 100000
# แล้วจัดทำ px.choropleth(..., color='rate')

<details>
<summary><b>แนวโค้ด (โครงสำหรับเติมช่องว่าง ____ ให้สมบูรณ์)</b></summary>

```python
pop = pd.read_csv('th_province_population.csv', encoding='utf-8-sig')

prov = df.groupby('PROV_NAME', as_index=False)['COMPLAIN_ALL'].sum()
prov = prov.merge(pop, on='____')                       # จับคู่ด้วยชื่อจังหวัด
prov['rate'] = prov['COMPLAIN_ALL'] / prov['____'] * ____

fig = px.choropleth(prov, geojson=geojson,
                    featureidkey='properties.pro_th', locations='PROV_NAME',
                    color='____', color_continuous_scale='Plasma')
fig.update_geos(fitbounds='locations', visible=False)
fig.update_layout(title='อัตราเบาะแสต่อประชากร 100,000 คน', height=600,
                  margin=dict(t=50, l=0, r=0, b=0))
fig.show()

print(prov.sort_values('rate', ascending=False)[['PROV_NAME', 'rate']].head())
```

</details>

<details>
<summary><b>เฉลย (โค้ดฉบับสมบูรณ์)</b></summary>

```python
pop = pd.read_csv('th_province_population.csv', encoding='utf-8-sig')

prov = df.groupby('PROV_NAME', as_index=False)['COMPLAIN_ALL'].sum()
prov = prov.merge(pop, on='PROV_NAME')
prov['rate'] = prov['COMPLAIN_ALL'] / prov['population'] * 100000

fig = px.choropleth(prov, geojson=geojson,
                    featureidkey='properties.pro_th', locations='PROV_NAME',
                    color='rate', color_continuous_scale='Plasma')
fig.update_geos(fitbounds='locations', visible=False)
fig.update_layout(title='อัตราเบาะแสต่อประชากร 100,000 คน', height=600,
                  margin=dict(t=50, l=0, r=0, b=0))
fig.show()

print(prov.sort_values('rate', ascending=False)[['PROV_NAME', 'rate']].head())
```

</details>

---
# ส่วนที่ 3 — บทเรียนการสร้างแบบจำลอง

ในส่วนก่อนหน้าเป็นเพียงการนำเสนอข้อมูล ในส่วนนี้จะเป็นการให้ระบบเรียนรู้รูปแบบจากข้อมูล
ซึ่งประกอบด้วยแบบจำลอง 2 ประเภท

| ประเภท | หน้าที่ | มีคำตอบให้เรียนรู้หรือไม่ |
|---|---|---|
| Clustering (K-Means) | จัดกลุ่มข้อมูลที่มีลักษณะคล้ายกันไว้ด้วยกัน | ไม่มี (unsupervised) |
| Regression (Linear) | ทำนายค่าตัวเลข จากตัวแปรต้นไปสู่ตัวแปรตาม | มี (supervised) |

### บทเรียนที่ 1 — การจัดกลุ่มด้วย K-Means

**Clustering** คือการให้ระบบจัดกลุ่มข้อมูลที่มีลักษณะคล้ายกันไว้ด้วยกัน โดยไม่ต้องกำหนดคำตอบล่วงหน้า (unsupervised)

**ขั้นตอนมาตรฐานประกอบด้วย 3 ขั้น**
1. **สร้างตัวแปร (feature)** ที่ใช้อธิบายแต่ละจังหวัด
2. **ปรับมาตราส่วน (`StandardScaler`)** ซึ่งมีความสำคัญมาก เนื่องจากหากตัวแปรหนึ่งมีค่าในหลักพัน อีกตัวแปรมีค่าในหลักหน่วย ตัวแปรที่มีค่ามากจะมีอิทธิพลเกินจริง จึงต้องปรับให้อยู่ในระดับเดียวกัน
3. **เรียกใช้ `KMeans(n_clusters=k).fit_predict(X)`** เพื่อกำหนดกลุ่มของแต่ละจังหวัด

**ตัวอย่าง:** จัดกลุ่มจังหวัดด้วยตัวแปร 2 ตัว คือปริมาณเบาะแส (log) และอัตราการพบพฤติการณ์ ซึ่งสามารถแสดงบนระนาบ 2 มิติได้โดยตรง

In [ ]:
# 1) สร้างตัวแปรรายจังหวัด 2 ตัว
g = df.groupby('PROV_NAME').agg(tips=('COMPLAIN_ALL','sum'),
                                found=('ALL1','sum'),
                                checked=('ALL123','sum')).reset_index()
g = g[g['tips'] >= 100]                          # เลือกเฉพาะจังหวัดที่มีข้อมูลเพียงพอ
g['log_tips']   = np.log1p(g['tips'])            # ปริมาณ (ใช้ log ลดอิทธิพลของค่าที่สูงผิดปกติ)
g['found_rate'] = g['found'] / g['checked']      # อัตราการตรวจพบพฤติการณ์
feats = ['log_tips', 'found_rate']

# 2) ปรับมาตราส่วนให้ตัวแปรทั้งสองอยู่ในระดับเดียวกัน
X = StandardScaler().fit_transform(g[feats])

# 3) จัดกลุ่มด้วย K-Means จำนวน 3 กลุ่ม
km = KMeans(n_clusters=3, random_state=42, n_init=10)
g['cluster'] = km.fit_predict(X).astype(str)

# นำเสนอผล: เนื่องจากมีตัวแปรเพียง 2 ตัว จึงแสดงบนระนาบจริงได้ (สี = กลุ่ม)
fig = px.scatter(g, x='log_tips', y='found_rate', color='cluster',
                 hover_name='PROV_NAME', size='tips',
                 title='การจัดกลุ่มจังหวัดด้วย K-Means (3 กลุ่ม จากตัวแปร 2 ตัว)')
fig.show()

> **ข้อสังเกตและการตีความ**  
> ข้อมูลถูกจำแนกออกเป็น 3 กลุ่มอย่างชัดเจน โดยแบ่งตามแกนนอน (ปริมาณเบาะแส) และแกนตั้ง (อัตราการพบพฤติการณ์) เป็นหลัก
> จะเห็นกลุ่มที่มีปริมาณสูง (ด้านขวา เช่น จังหวัดขนาดใหญ่) แยกจากกลุ่มที่มีปริมาณน้อย (ด้านซ้าย) และมีกลุ่มที่อัตราการพบพฤติการณ์สูงหรือต่ำแตกต่างกัน
> ผลลัพธ์นี้มีความสมเหตุสมผล เนื่องจาก K-Means จะจัดจุดที่อยู่ใกล้กันไว้กลุ่มเดียวกัน จังหวัดที่มีลักษณะคล้ายกันจึงถูกจัดอยู่ในกลุ่มเดียวกัน

> กรณีที่มีตัวแปรมากกว่า 2 ตัว จะไม่สามารถแสดงบนระนาบ 2 มิติได้โดยตรง จึงต้องใช้เทคนิค **PCA**
> เพื่อลดข้อมูลให้เหลือ 2 แกนหลัก (PC1, PC2) สำหรับการนำเสนอ ซึ่งจะได้ฝึกปฏิบัติในแบบฝึกปฏิบัติที่ 6

### บทเรียนที่ 2 — การถดถอยด้วย Linear Regression

**Regression** คือการทำนายค่าตัวเลขจากตัวแปรที่มีอยู่ โดยมีคำตอบจริงให้ระบบเรียนรู้ (supervised)

- **X (ตัวแปรต้น)** คือสิ่งที่ใช้ในการทำนาย (ต้องอยู่ในรูปตาราง 2 มิติ จึงใช้ `[[...]]`)
- **y (ตัวแปรตาม)** คือค่าที่ต้องการทำนาย

**Linear Regression** จะกำหนดเส้นตรงที่เหมาะสมกับข้อมูลมากที่สุด ซึ่งเส้นดังกล่าวคือ **สมการ**

> **`y  =  (ความชัน × x)  +  จุดตัดแกน`**

(ความชัน หมายถึง เมื่อ x เพิ่มขึ้น 1 หน่วย y จะเปลี่ยนแปลงเท่าใด ส่วนจุดตัดแกน หมายถึงค่า y เมื่อ x เท่ากับ 0)

ประเมินความแม่นยำด้วย **R²** (มีค่า 0–1 ยิ่งเข้าใกล้ 1 ยิ่งดี) และ **MAE** (ค่าความคลาดเคลื่อนเฉลี่ย ยิ่งน้อยยิ่งดี)

**ตัวอย่าง:** ทำนายจำนวนที่พบพฤติการณ์ (`ALL1`) จากจำนวนที่ตรวจสอบทั้งหมด (`ALL123`) ในระดับจังหวัดต่อเดือน

In [ ]:
# 1) เตรียมข้อมูล: X = จำนวนที่ตรวจสอบ, y = จำนวนที่พบพฤติการณ์
X = df[['ALL123']]     # ตัวแปรต้น (2 มิติ จึงใช้ [[...]])
y = df['ALL1']         # ตัวแปรตาม

# 2) สร้างแบบจำลองแล้วเรียนรู้จากข้อมูล
model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)

# 3) แสดงค่าสัมประสิทธิ์ของสมการที่แบบจำลองเรียนรู้ได้
slope     = model.coef_[0]        # ความชัน
intercept = model.intercept_      # จุดตัดแกน y
print(f'สมการ:  ALL1 = {slope:.3f} x ALL123 + {intercept:.2f}')
print(f'ความชัน {slope:.3f} หมายความว่า เมื่อตรวจสอบเพิ่ม 100 เรื่อง จะพบพฤติการณ์เพิ่มขึ้นประมาณ {slope*100:.0f} เรื่อง')
print()

# 4) ประเมินผลแบบจำลอง
print(f'R2  = {r2_score(y, y_pred):.3f}')
print(f'MAE = {mean_absolute_error(y, y_pred):.1f} เรื่อง')

**การนำแบบจำลองไปใช้ทำนายค่าใหม่** — เมื่อได้สมการแล้ว สามารถกำหนดค่าใหม่เพื่อให้แบบจำลองทำนายได้

In [ ]:
# สมมติมีจังหวัด/เดือนที่ตรวจสอบ 100 และ 300 เรื่อง แบบจำลองจะทำนายจำนวนที่พบพฤติการณ์เท่าใด
new_data = pd.DataFrame({'ALL123': [100, 300]})   # ต้องอยู่ในรูปตาราง 2 มิติเช่นเดียวกับตอนเรียนรู้
pred = model.predict(new_data)

for checked, found in zip(new_data['ALL123'], pred):
    print(f'ตรวจสอบ {checked} เรื่อง → ทำนายว่าพบพฤติการณ์ประมาณ {found:.0f} เรื่อง')

In [ ]:
# แสดงข้อมูลจริง (แต่ละจังหวัดต่อเดือน) พร้อมเส้นทำนายจากสมการ
order = df['ALL123'].argsort()      # เรียงค่า x เพื่อให้เส้นต่อเนื่อง
fig = px.scatter(df, x='ALL123', y='ALL1', opacity=0.4,
                 title='การทำนายจำนวนที่พบพฤติการณ์ จากจำนวนที่ตรวจสอบ',
                 labels={'ALL123':'ตรวจสอบทั้งหมด (เรื่อง)', 'ALL1':'พบพฤติการณ์ (เรื่อง)'})
fig.add_trace(go.Scatter(x=df['ALL123'].iloc[order], y=y_pred[order],
                         mode='lines', name='เส้นทำนาย', line=dict(color='red', width=3)))
fig.show()

> **ข้อสังเกตและการตีความ**  
> ข้อมูลเรียงตัวสูงขึ้นไปทางขวา และเส้นทำนาย (สมการ) มีความชันเป็นบวก แสดงว่ายิ่งตรวจสอบมาก ยิ่งพบพฤติการณ์มาก ซึ่งมีความสมเหตุสมผล
> ความชันประมาณ 0.18 หมายความว่าเมื่อตรวจสอบเพิ่ม 100 เรื่อง จะพบพฤติการณ์เพิ่มประมาณ 18 เรื่อง สอดคล้องกับอัตราการพบประมาณร้อยละ 33 ที่พบในตัวอย่างที่ 2
> อย่างไรก็ตาม ค่า R² ประมาณ 0.48 แสดงว่าสมการอธิบายข้อมูลได้ประมาณครึ่งหนึ่ง ข้อมูลยังกระจายรอบเส้นพอสมควร แสดงว่าจำนวนที่ตรวจสอบเพียงอย่างเดียวยังไม่เพียงพอ ยังขึ้นอยู่กับคุณภาพของเบาะแสในแต่ละพื้นที่ด้วย

---
# ส่วนที่ 4 — แบบฝึกปฏิบัติการสร้างแบบจำลอง

ในส่วนนี้เป็นการฝึกปฏิบัติด้วยตนเอง โดยต่อยอดจากตัวอย่างข้างต้น (เพิ่มตัวแปร หรือเปลี่ยนตัวแปร)

## แบบฝึกปฏิบัติที่ 6 — การจัดกลุ่มด้วยตัวแปร 3 ตัว ร่วมกับ PCA (ระดับสูง)

**โจทย์:** ต่อยอดจากบทเรียนที่ 1 โดยใช้ตัวแปร 3 ตัว และจัดกลุ่มเป็น 4 กลุ่ม
- `log_vol` (ปริมาณ) · `found_rate` (อัตราการพบ) · `verify_rate` = ALL123/COMPLAIN_ALL (สัดส่วนที่ได้รับการตรวจสอบ)

เนื่องจากมีตัวแปร 3 ตัวซึ่งไม่สามารถแสดงบนระนาบ 2 มิติได้โดยตรง จึงต้องใช้ **PCA** ลดข้อมูลเหลือ 2 มิติก่อนนำเสนอ

**สิ่งที่ต้องดำเนินการ:** ปรับมาตราส่วนด้วย `StandardScaler` + สร้าง `KMeans(n_clusters=4)` + เรียก `fit_predict`

In [ ]:
# จัดเตรียมตัวแปร 3 ตัวให้แล้ว — ให้ดำเนินการต่อในส่วนของแบบจำลอง
g = df.groupby('PROV_NAME').agg(tips=('COMPLAIN_ALL','sum'),
                                found=('ALL1','sum'),
                                checked=('ALL123','sum')).reset_index()
g = g[g['tips'] >= 100]
g['log_vol']     = np.log1p(g['tips'])
g['found_rate']  = g['found']   / g['checked']
g['verify_rate'] = g['checked'] / g['tips']
features = ['log_vol', 'found_rate', 'verify_rate']

# ดำเนินการต่อ: X = StandardScaler().fit_transform(g[features])
# km = KMeans(n_clusters=4, random_state=42, n_init=10)
# g['cluster'] = km.fit_predict(X).astype(str)
# แล้วใช้ส่วน PCA ในแนวโค้ดเพื่อนำเสนอผล

<details>
<summary><b>แนวโค้ด (โครงสำหรับเติมช่องว่าง ____ ให้สมบูรณ์)</b></summary>

```python
g = df.groupby('PROV_NAME').agg(tips=('COMPLAIN_ALL','sum'),
                                found=('ALL1','sum'),
                                checked=('ALL123','sum')).reset_index()
g = g[g['tips'] >= 100]
g['log_vol']     = np.log1p(g['tips'])
g['found_rate']  = g['found']   / g['checked']
g['verify_rate'] = g['checked'] / g['tips']
features = ['log_vol', 'found_rate', 'verify_rate']

X = StandardScaler().fit_transform(g[____])         # ระบุ features
km = KMeans(n_clusters=____, random_state=42, n_init=10)
g['cluster'] = km.____(X).astype(str)               # fit_predict

# ลดข้อมูล 3 มิติเหลือ 2 มิติเพื่อนำเสนอ
coords = PCA(n_components=2, random_state=42).fit_transform(X)
g['PC1'], g['PC2'] = coords[:,0], coords[:,1]
fig = px.scatter(g, x='PC1', y='PC2', color='cluster', size='tips',
                 hover_name='PROV_NAME', title='K-Means 4 กลุ่ม (ตัวแปร 3 ตัว ร่วมกับ PCA)')
fig.show()
print(g.groupby('cluster')[features].mean().round(3))
```

</details>

<details>
<summary><b>เฉลย (โค้ดฉบับสมบูรณ์)</b></summary>

```python
g = df.groupby('PROV_NAME').agg(tips=('COMPLAIN_ALL','sum'),
                                found=('ALL1','sum'),
                                checked=('ALL123','sum')).reset_index()
g = g[g['tips'] >= 100]
g['log_vol']     = np.log1p(g['tips'])
g['found_rate']  = g['found']   / g['checked']
g['verify_rate'] = g['checked'] / g['tips']
features = ['log_vol', 'found_rate', 'verify_rate']

X = StandardScaler().fit_transform(g[features])
km = KMeans(n_clusters=4, random_state=42, n_init=10)
g['cluster'] = km.fit_predict(X).astype(str)

coords = PCA(n_components=2, random_state=42).fit_transform(X)
g['PC1'], g['PC2'] = coords[:,0], coords[:,1]
fig = px.scatter(g, x='PC1', y='PC2', color='cluster', size='tips',
                 hover_name='PROV_NAME', title='K-Means 4 กลุ่ม (ตัวแปร 3 ตัว ร่วมกับ PCA)')
fig.show()
print(g.groupby('cluster')[features].mean().round(3))
```

</details>

## แบบฝึกปฏิบัติที่ 7 — การถดถอย: ประชากรอธิบายจำนวนเบาะแสได้เพียงใด

**ประเด็นศึกษา:** จังหวัดที่มีประชากรมาก จะมีจำนวนเบาะแสมากตามไปด้วยหรือไม่ และอธิบายได้ในสัดส่วนเท่าใด

ต่อยอดจากบทเรียนที่ 2 โดยเปลี่ยนตัวแปรเป็น
- **X:** `population` (จำนวนประชากร) · **y:** `COMPLAIN_ALL` (จำนวนเบาะแสรวมรายจังหวัด)

**สิ่งที่ต้องดำเนินการ:** สร้าง `LinearRegression()` → `.fit` → `.predict` → `r2_score` แล้วทดลองทำนายจังหวัดสมมติที่มีประชากร 1,000,000 คน

In [ ]:
# จัดเตรียม X และ y ให้แล้ว
pop  = pd.read_csv('th_province_population.csv', encoding='utf-8-sig')
data = df.groupby('PROV_NAME', as_index=False)['COMPLAIN_ALL'].sum().merge(pop, on='PROV_NAME')
X = data[['population']]      # 2 มิติ
y = data['COMPLAIN_ALL']

# ดำเนินการต่อ: model = LinearRegression() ; model.fit(X, y) ; y_pred = model.predict(X)
# print(r2_score(y, y_pred))
# ทำนายจังหวัดที่มีประชากร 1,000,000 คน ด้วย model.predict(pd.DataFrame({'population':[1_000_000]}))

<details>
<summary><b>แนวโค้ด (โครงสำหรับเติมช่องว่าง ____ ให้สมบูรณ์)</b></summary>

```python
pop  = pd.read_csv('th_province_population.csv', encoding='utf-8-sig')
data = df.groupby('PROV_NAME', as_index=False)['COMPLAIN_ALL'].sum().merge(pop, on='PROV_NAME')
X = data[['population']]
y = data['COMPLAIN_ALL']

model  = ____()             # สร้างแบบจำลอง
model.____(X, y)            # เรียนรู้
y_pred = model.____(X)      # ทำนาย

print(f'R2  = {____(y, y_pred):.3f}')
print(f'MAE = {mean_absolute_error(y, y_pred):,.0f} เรื่อง')

# ทำนายจังหวัดสมมติที่มีประชากร 1,000,000 คน
guess = model.predict(pd.DataFrame({'population': [1_000_000]}))
print(f'ประชากร 1,000,000 คน → ทำนายเบาะแสประมาณ {guess[0]:,.0f} เรื่อง')

fig = px.scatter(data, x='population', y='COMPLAIN_ALL', hover_name='PROV_NAME',
                 title='ประชากร เทียบกับ จำนวนเบาะแส')
fig.show()
```

</details>

<details>
<summary><b>เฉลย (โค้ดฉบับสมบูรณ์)</b></summary>

```python
pop  = pd.read_csv('th_province_population.csv', encoding='utf-8-sig')
data = df.groupby('PROV_NAME', as_index=False)['COMPLAIN_ALL'].sum().merge(pop, on='PROV_NAME')
X = data[['population']]
y = data['COMPLAIN_ALL']

model  = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)

print(f'R2  = {r2_score(y, y_pred):.3f}')
print(f'MAE = {mean_absolute_error(y, y_pred):,.0f} เรื่อง')

guess = model.predict(pd.DataFrame({'population': [1_000_000]}))
print(f'ประชากร 1,000,000 คน → ทำนายเบาะแสประมาณ {guess[0]:,.0f} เรื่อง')

fig = px.scatter(data, x='population', y='COMPLAIN_ALL', hover_name='PROV_NAME',
                 title='ประชากร เทียบกับ จำนวนเบาะแส')
fig.show()
```

</details>

---
## สรุปและคำถามเพื่อการอภิปราย

เนื้อหาที่ได้ฝึกปฏิบัติในเอกสารฉบับนี้ ประกอบด้วย
- **การนำเสนอข้อมูล (Visualization):** Histogram, Bar chart, Line chart และแผนที่ Choropleth (จากไฟล์ GeoJSON)
- **การจัดกลุ่ม (Clustering ด้วย K-Means):** จัดกลุ่มจังหวัดตามลักษณะของเบาะแส ร่วมกับการใช้ PCA เมื่อมีตัวแปรมากกว่า 2 ตัว
- **การถดถอย (Regression):** การสร้างสมการทำนาย การประเมินผลด้วย R² และ MAE และการทำนายค่าใหม่

**คำถามเพื่อการอภิปราย**
1. จังหวัดที่มีเบาะแสต่อประชากรสูงสุด (แบบฝึกปฏิบัติที่ 5) แตกต่างจากจังหวัดที่มีเบาะแสรวมสูงสุด (แบบฝึกปฏิบัติที่ 2) หรือไม่ เพราะเหตุใด
2. แต่ละกลุ่มจากการจัดกลุ่มด้วย K-Means (แบบฝึกปฏิบัติที่ 6) มีลักษณะเด่นอย่างไร ควรตั้งชื่อกลุ่มแต่ละกลุ่มว่าอย่างไรจากค่าเฉลี่ยที่แสดงผล
3. ค่า R² จากแบบฝึกปฏิบัติที่ 7 สื่อความหมายอย่างไร หากต้องการเพิ่มความแม่นยำในการทำนาย ควรเพิ่มตัวแปรใดอีก